# 012 Top-m decoding bumps by normalized component ratio — matched comparison

Follow-up to notebook 11.

This notebook loads saved `topm_summary.csv` files and asks whether the top-m decoding “bump” occurs at different normalized component ratios for spatial vs temporal AA.

Key improvements in this version:

1. **Spatial and temporal are visually distinct**
   - spatial = solid line + circle markers
   - temporal = dashed line + square markers

2. **All available data are plotted**
   - raw curves use all available K values

3. **Matched spatial/temporal comparisons are computed**
   - for each condition and top-m value, spatial and temporal points are paired by nearest normalized component ratio
   - matched comparisons are summarized and plotted separately


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

FIT_SCOPE = "across"  # "across" or "within"

DECODE_DIR_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}"

ANALYSIS_TYPES = ["spatial", "temporal"]
CONDITIONS = ["intact", "word", "rest"]

DIM_MAP = {
    "spatial": 700,
    "temporal": 300,
}

# None = plot all available top_m values
TOP_M_VALUES_TO_PLOT = None
# Example:
# TOP_M_VALUES_TO_PLOT = [1, 3, 5, 10]

# Optional ratio window
RATIO_MIN = 0
RATIO_MAX = .2
# Example:
# RATIO_MIN = 0.05
# RATIO_MAX = 0.60

# Optional K filters
K_VALUES_BY_ANALYSIS = {
    "spatial": None,
    "temporal": None,
}

# Peak detection
BUMP_MODE = "max"  # "max" or "local"
ANNOTATE_PEAKS = True

# Matched comparison settings
# "temporal_to_spatial": each temporal point gets nearest spatial point
# "spatial_to_temporal": each spatial point gets nearest temporal point
# "symmetric_best": non-duplicated nearest pairs
MATCH_METHOD = "symmetric_best"

# Maximum allowed absolute difference in normalized component ratio.
# Increase this if you want more matched points from sparse K grids.
MAX_RATIO_DIFF = 0.015

# If True, create a matched version of the top-m curves
PLOT_MATCHED_CURVES = True

# Plotting
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "012_topm_bump_normalized_component_ratio_v2_matched"
SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

FIGSIZE = (9.0, 5.4)
YLIM = None

COND_COLORS = {
    "intact": "purple",
    "word": "green",
    "rest": "black",
}

ANALYSIS_LINESTYLES = {
    "spatial": "-",
    "temporal": ":",
}

ANALYSIS_MARKERS = {
    "spatial": "o",
    "temporal": "s",
}


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

_fig_counter = 0

def _safe_name(name):
    name = str(name).replace(" ", "_").replace("/", "-").replace("|", "_")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def save_current_fig(name):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    _fig_counter += 1
    out = FIG_DIR / f"{_fig_counter:03d}_{_safe_name(name)}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def add_two_part_legend(ax):
    """
    Creates clear, non-redundant legends:
    - color legend = condition
    - line/marker legend = analysis type
    """
    condition_handles = [
        Line2D([0], [0], color=COND_COLORS[c], lw=2.5, label=c)
        for c in CONDITIONS
    ]

    analysis_handles = [
        Line2D(
            [0], [0],
            color="black",
            linestyle=ANALYSIS_LINESTYLES[a],
            marker=ANALYSIS_MARKERS[a],
            lw=2.5,
            label=a
        )
        for a in ANALYSIS_TYPES
    ]

    leg1 = ax.legend(handles=condition_handles, title="Condition", frameon=False, loc="upper left")
    ax.add_artist(leg1)
    ax.legend(handles=analysis_handles, title="Analysis", frameon=False, loc="upper right")

print("Figure directory:", FIG_DIR)


## Load saved top-m summaries

In [ ]:
def standardize_topm_df(df, analysis_type, fit_scope):
    df = df.copy()

    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "err" not in df.columns:
        rename["sem_accuracy"] = "err"
    if "std_accuracy" in df.columns and "std" not in df.columns:
        rename["std_accuracy"] = "std"
    if "sem" in df.columns and "err" not in df.columns:
        rename["sem"] = "err"

    df = df.rename(columns=rename)

    if "analysis_type" not in df.columns:
        df["analysis_type"] = analysis_type
    if "fit_scope" not in df.columns:
        df["fit_scope"] = fit_scope

    df["analysis_type"] = df["analysis_type"].astype(str)
    df["fit_scope"] = df["fit_scope"].astype(str)
    df["condition"] = df["condition"].astype(str)
    df["K"] = df["K"].astype(int)
    df["top_m"] = df["top_m"].astype(int)

    df = df[
        (df["analysis_type"] == analysis_type) &
        (df["fit_scope"] == fit_scope)
    ].copy()

    k_filter = K_VALUES_BY_ANALYSIS.get(analysis_type, None)
    if k_filter is not None:
        df = df[df["K"].isin(k_filter)].copy()

    return df


def load_topm_summary(analysis_type):
    d = Path(DECODE_DIR_TEMPLATE.format(
        analysis_type=analysis_type,
        fit_scope=FIT_SCOPE
    ))
    path = d / "topm_summary.csv"

    if not path.exists():
        print("Missing:", path)
        return None

    print("Loaded:", path)
    df = pd.read_csv(path)
    return standardize_topm_df(df, analysis_type, FIT_SCOPE)


dfs = []
for analysis_type in ANALYSIS_TYPES:
    df = load_topm_summary(analysis_type)
    if df is not None and len(df):
        dfs.append(df)

topm_df = pd.concat(dfs, ignore_index=True)

topm_df["axis_dim"] = topm_df["analysis_type"].map(DIM_MAP)
topm_df["normalized_component_ratio"] = topm_df["K"] / topm_df["axis_dim"]

if TOP_M_VALUES_TO_PLOT is not None:
    topm_df = topm_df[topm_df["top_m"].isin(TOP_M_VALUES_TO_PLOT)].copy()

if RATIO_MIN is not None:
    topm_df = topm_df[topm_df["normalized_component_ratio"] >= RATIO_MIN].copy()
if RATIO_MAX is not None:
    topm_df = topm_df[topm_df["normalized_component_ratio"] <= RATIO_MAX].copy()

print("Rows:", len(topm_df))
print("Top-m values:", sorted(topm_df["top_m"].unique()))
display(topm_df.head())

print("\nAvailable K values by analysis / top_m / condition:")
display(
    topm_df.groupby(["analysis_type", "top_m", "condition"])["K"]
    .apply(lambda x: sorted(x.unique()))
    .reset_index()
    .head(30)
)


## Peak / bump detection using all available data

In [ ]:
def get_err_col(df):
    for col in ["err", "sem", "stderr", "se", "sem_accuracy"]:
        if col in df.columns:
            return col
    return None


def find_bump(sub, mode="max"):
    sub = sub.sort_values("normalized_component_ratio").copy()

    if len(sub) == 0:
        return None

    if mode == "max" or len(sub) < 3:
        return sub.loc[sub["mean"].idxmax()]

    if mode == "local":
        y = sub["mean"].to_numpy()
        local_idx = []
        for i in range(1, len(y) - 1):
            if y[i] >= y[i - 1] and y[i] >= y[i + 1]:
                local_idx.append(i)

        if len(local_idx) == 0:
            return sub.loc[sub["mean"].idxmax()]

        local_sub = sub.iloc[local_idx]
        return local_sub.loc[local_sub["mean"].idxmax()]

    raise ValueError("BUMP_MODE must be 'max' or 'local'.")


def build_bump_summary(df):
    rows = []

    for (analysis_type, condition, top_m), sub in df.groupby(["analysis_type", "condition", "top_m"]):
        bump = find_bump(sub, mode=BUMP_MODE)
        if bump is None:
            continue

        rows.append({
            "analysis_type": analysis_type,
            "condition": condition,
            "top_m": int(top_m),
            "K_at_bump": int(bump["K"]),
            "ratio_at_bump": float(bump["normalized_component_ratio"]),
            "mean_at_bump": float(bump["mean"]),
            "axis_dim": int(bump["axis_dim"]),
        })

    return pd.DataFrame(rows).sort_values(["condition", "top_m", "analysis_type"])


bump_summary = build_bump_summary(topm_df)
display(bump_summary)


# Plot 1: Top-m decoding curves using all available K values

In [ ]:
err_col = get_err_col(topm_df)
top_m_values = sorted(topm_df["top_m"].unique())

for top_m in top_m_values:
    fig, ax = plt.subplots(figsize=FIGSIZE)

    sub_m = topm_df[topm_df["top_m"] == top_m].copy()
    bump_m = bump_summary[bump_summary["top_m"] == top_m].copy()

    for analysis_type in ANALYSIS_TYPES:
        for condition in CONDITIONS:
            sub = sub_m[
                (sub_m["analysis_type"] == analysis_type) &
                (sub_m["condition"] == condition)
            ].sort_values("normalized_component_ratio")

            if len(sub) == 0:
                continue

            yerr = sub[err_col] if err_col is not None else None

            ax.errorbar(
                sub["normalized_component_ratio"],
                sub["mean"],
                yerr=yerr,
                marker=ANALYSIS_MARKERS[analysis_type],
                capsize=4,
                linewidth=2.2,
                color=COND_COLORS[condition],
                linestyle=ANALYSIS_LINESTYLES[analysis_type],
            )

    if ANNOTATE_PEAKS and len(bump_m):
        ymin, ymax = ax.get_ylim()
        yrange = ymax - ymin

        for _, row in bump_m.iterrows():
            ax.scatter(
                row["ratio_at_bump"],
                row["mean_at_bump"],
                s=95,
                facecolors="none",
                edgecolors=COND_COLORS[row["condition"]],
                linewidths=2,
                zorder=10,
            )

            ax.text(
                row["ratio_at_bump"],
                row["mean_at_bump"] + 0.03 * yrange,
                f"{row['analysis_type'][0].upper()} K={int(row['K_at_bump'])}",
                fontsize=8,
                ha="center",
                va="bottom",
                color=COND_COLORS[row["condition"]],
            )

    if YLIM is not None:
        ax.set_ylim(*YLIM)

    ax.set_xlabel("Normalized component ratio (K / axis dimensionality)")
    ax.set_ylabel("Top-m decoding accuracy")
    ax.set_title(f"Top-{top_m} decoding bumps by normalized ratio | {FIT_SCOPE}\nall available K values")
    add_two_part_legend(ax)
    plt.tight_layout()
    save_current_fig(f"top{top_m}_allK_decoding_bumps_normalized_ratio_{FIT_SCOPE}")
    plt.show()
    plt.close()


## Build matched spatial/temporal comparisons

In [ ]:
def candidate_pairs_for_group(df, condition, top_m):
    spatial = df[
        (df["analysis_type"] == "spatial") &
        (df["condition"] == condition) &
        (df["top_m"] == top_m)
    ].copy()

    temporal = df[
        (df["analysis_type"] == "temporal") &
        (df["condition"] == condition) &
        (df["top_m"] == top_m)
    ].copy()

    rows = []

    for _, s in spatial.iterrows():
        for _, t in temporal.iterrows():
            ratio_diff = abs(
                s["normalized_component_ratio"] -
                t["normalized_component_ratio"]
            )

            if ratio_diff <= MAX_RATIO_DIFF:
                rows.append({
                    "condition": condition,
                    "top_m": int(top_m),
                    "K_spatial": int(s["K"]),
                    "K_temporal": int(t["K"]),
                    "ratio_spatial": float(s["normalized_component_ratio"]),
                    "ratio_temporal": float(t["normalized_component_ratio"]),
                    "ratio_mid": float((s["normalized_component_ratio"] + t["normalized_component_ratio"]) / 2),
                    "ratio_diff": float(ratio_diff),
                    "mean_spatial": float(s["mean"]),
                    "mean_temporal": float(t["mean"]),
                    "diff_temporal_minus_spatial": float(t["mean"] - s["mean"]),
                    "diff_spatial_minus_temporal": float(s["mean"] - t["mean"]),
                    "err_spatial": float(s[err_col]) if err_col is not None else np.nan,
                    "err_temporal": float(t[err_col]) if err_col is not None else np.nan,
                })

    return pd.DataFrame(rows)


def select_pairs(candidates, method):
    if len(candidates) == 0:
        return candidates

    candidates = candidates.sort_values("ratio_diff").copy()

    if method == "temporal_to_spatial":
        return (
            candidates.groupby(["condition", "top_m", "K_temporal"], as_index=False)
            .head(1)
            .reset_index(drop=True)
        )

    if method == "spatial_to_temporal":
        return (
            candidates.groupby(["condition", "top_m", "K_spatial"], as_index=False)
            .head(1)
            .reset_index(drop=True)
        )

    if method == "symmetric_best":
        keep = []
        used_spatial = set()
        used_temporal = set()

        for _, row in candidates.iterrows():
            s_key = (row["condition"], int(row["top_m"]), int(row["K_spatial"]))
            t_key = (row["condition"], int(row["top_m"]), int(row["K_temporal"]))

            if s_key in used_spatial or t_key in used_temporal:
                continue

            keep.append(row)
            used_spatial.add(s_key)
            used_temporal.add(t_key)

        if len(keep) == 0:
            return pd.DataFrame(columns=candidates.columns)

        return pd.DataFrame(keep).reset_index(drop=True)

    raise ValueError("Unknown MATCH_METHOD")


candidate_list = []
for condition in CONDITIONS:
    for top_m in sorted(topm_df["top_m"].unique()):
        candidate_list.append(candidate_pairs_for_group(topm_df, condition, top_m))

candidate_pairs = pd.concat(candidate_list, ignore_index=True)
matched_df = select_pairs(candidate_pairs, MATCH_METHOD)

print("Candidate pairs:", len(candidate_pairs))
print("Matched pairs:", len(matched_df))
display(matched_df.head(30))

print("\nMatched points by condition/top_m:")
display(
    matched_df.groupby(["condition", "top_m"])
    .size()
    .reset_index(name="n_matched_pairs")
)


# Plot 2: Matched top-m curves

In [ ]:
if PLOT_MATCHED_CURVES:
    for top_m in sorted(matched_df["top_m"].unique()):
        fig, ax = plt.subplots(figsize=FIGSIZE)

        sub_m = matched_df[matched_df["top_m"] == top_m].copy()

        for condition in CONDITIONS:
            sub = sub_m[sub_m["condition"] == condition].sort_values("ratio_mid")

            if len(sub) == 0:
                continue

            # spatial matched curve
            ax.errorbar(
                sub["ratio_mid"],
                sub["mean_spatial"],
                yerr=sub["err_spatial"] if err_col is not None else None,
                marker=ANALYSIS_MARKERS["spatial"],
                capsize=4,
                linewidth=2.2,
                color=COND_COLORS[condition],
                linestyle=ANALYSIS_LINESTYLES["spatial"],
            )

            # temporal matched curve
            ax.errorbar(
                sub["ratio_mid"],
                sub["mean_temporal"],
                yerr=sub["err_temporal"] if err_col is not None else None,
                marker=ANALYSIS_MARKERS["temporal"],
                capsize=4,
                linewidth=2.2,
                color=COND_COLORS[condition],
                linestyle=ANALYSIS_LINESTYLES["temporal"],
            )

        if YLIM is not None:
            ax.set_ylim(*YLIM)

        ax.set_xlabel("Matched normalized component ratio")
        ax.set_ylabel("Top-m decoding accuracy")
        ax.set_title(
            f"Top-{top_m} matched spatial/temporal comparison | {FIT_SCOPE}\n"
            f"method={MATCH_METHOD}, max ratio diff={MAX_RATIO_DIFF}"
        )
        add_two_part_legend(ax)
        plt.tight_layout()
        save_current_fig(f"top{top_m}_matched_curves_{FIT_SCOPE}_{MATCH_METHOD}")
        plt.show()
        plt.close()
else:
    print("PLOT_MATCHED_CURVES=False")


# Plot 3: Spatial minus temporal decoding at matched ratios

In [ ]:
for top_m in sorted(matched_df["top_m"].unique()):
    fig, ax = plt.subplots(figsize=FIGSIZE)

    sub_m = matched_df[matched_df["top_m"] == top_m].copy()

    for condition in CONDITIONS:
        sub = sub_m[sub_m["condition"] == condition].sort_values("ratio_mid")

        if len(sub) == 0:
            continue

        diff_se = np.sqrt(sub["err_spatial"]**2 + sub["err_temporal"]**2) if err_col is not None else None

        ax.errorbar(
            sub["ratio_mid"],
            sub["diff_spatial_minus_temporal"],
            yerr=diff_se,
            marker="o",
            capsize=4,
            linewidth=2.2,
            color=COND_COLORS[condition],
            label=condition,
        )

    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Matched normalized component ratio")
    ax.set_ylabel("Spatial minus temporal top-m decoding")
    ax.set_title(f"Top-{top_m}: spatial - temporal at matched ratios | {FIT_SCOPE}")
    ax.legend(frameon=False)
    plt.tight_layout()
    save_current_fig(f"top{top_m}_matched_spatial_minus_temporal_{FIT_SCOPE}_{MATCH_METHOD}")
    plt.show()
    plt.close()


# Plot 4: Bump location from all data vs matched data

In [ ]:
def build_matched_bump_summary(matched):
    rows = []

    for (condition, top_m), sub in matched.groupby(["condition", "top_m"]):
        sub = sub.sort_values("ratio_mid").copy()

        if len(sub) == 0:
            continue

        # spatial bump among matched points
        srow = sub.loc[sub["mean_spatial"].idxmax()]
        trow = sub.loc[sub["mean_temporal"].idxmax()]

        rows.append({
            "condition": condition,
            "top_m": int(top_m),
            "analysis_type": "spatial",
            "K_at_bump": int(srow["K_spatial"]),
            "ratio_at_bump": float(srow["ratio_mid"]),
            "mean_at_bump": float(srow["mean_spatial"]),
            "source": "matched",
        })

        rows.append({
            "condition": condition,
            "top_m": int(top_m),
            "analysis_type": "temporal",
            "K_at_bump": int(trow["K_temporal"]),
            "ratio_at_bump": float(trow["ratio_mid"]),
            "mean_at_bump": float(trow["mean_temporal"]),
            "source": "matched",
        })

    return pd.DataFrame(rows)


matched_bump_summary = build_matched_bump_summary(matched_df)

fig, ax = plt.subplots(figsize=FIGSIZE)

for source, alpha, marker_shift in [("all", 1.0, -0.08), ("matched", 0.65, 0.08)]:
    df_source = bump_summary.copy() if source == "all" else matched_bump_summary.copy()

    for analysis_type in ANALYSIS_TYPES:
        for condition in CONDITIONS:
            sub = df_source[
                (df_source["analysis_type"] == analysis_type) &
                (df_source["condition"] == condition)
            ].sort_values("top_m")

            if len(sub) == 0:
                continue

            x = sub["top_m"] + marker_shift

            ax.plot(
                x,
                sub["ratio_at_bump"],
                marker=ANALYSIS_MARKERS[analysis_type],
                linewidth=2.0,
                color=COND_COLORS[condition],
                linestyle=ANALYSIS_LINESTYLES[analysis_type],
                alpha=alpha,
                label=f"{source} | {analysis_type} | {condition}",
            )

ax.set_xlabel("Top-m archetypes included")
ax.set_ylabel("Normalized component ratio at bump")
ax.set_title(f"Bump location by top-m: all data vs matched data | {FIT_SCOPE}")
ax.legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout()
save_current_fig(f"bump_location_all_vs_matched_by_topm_{FIT_SCOPE}")
plt.show()
plt.close()

display(matched_bump_summary.head(20))


## Save tables

In [ ]:
out_bump = FIG_DIR / f"topm_bump_summary_allK_{FIT_SCOPE}.csv"
out_matched = FIG_DIR / f"topm_matched_pairs_{FIT_SCOPE}_{MATCH_METHOD}.csv"
out_matched_bump = FIG_DIR / f"topm_bump_summary_matched_{FIT_SCOPE}_{MATCH_METHOD}.csv"

bump_summary.to_csv(out_bump, index=False)
matched_df.to_csv(out_matched, index=False)
matched_bump_summary.to_csv(out_matched_bump, index=False)

print("Saved:", out_bump)
print("Saved:", out_matched)
print("Saved:", out_matched_bump)

display(bump_summary.head(30))
display(matched_df.head(30))
display(matched_bump_summary.head(30))
